In [46]:
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from pathlib import Path

In [47]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


## Semiconductor Peer Comparison: Normalized Price Performance
### Tracks normalized price movement (Jan 15 = 100) for NVDA, AMD, ASML, ARM, and SOXX from Jan 15 to Mar 15, 2025.
### Note: Normalized price sets each stock's value to 100 on Jan 15, 2025. All subsequent values show the percentage change relative to that starting point — so 90 means the stock dropped 10%, and 110 means it gained 10%. This allows fair comparison across stocks with very different absolute prices (e.g. ASML at ~$726 vs AMD at ~$120).
### R² shown on chart measures correlation strength — closer to 0 means weak relationship, closer to 1 means strong.

In [48]:
start = "2025-01-15"
end = "2025-03-15"

data_dir = Path.cwd()
if not (data_dir / "NVDA.csv").exists():
    data_dir = data_dir.parent

nvda = pd.read_csv(data_dir / "NVDA.csv")
nvda["ticker"] = "NVDA"
amd = pd.read_csv(data_dir / "AMD.csv")
amd["ticker"] = "AMD"
asml = pd.read_csv(data_dir / "ASML.csv")
asml["ticker"] = "ASML"
arm = pd.read_csv(data_dir / "ARM.csv")
arm["ticker"] = "ARM"
soxx = pd.read_csv(data_dir / "SOXX.csv")
soxx["ticker"] = "SOXX"
df = pd.concat([nvda, asml, arm, amd, soxx])

# filter date
df["date"] = pd.to_datetime(df["date"])
df = df[(df["date"] >= start) & (df["date"] <= end)]

# date = 15, normalized price
df["norm_price"] = df["close"] / df.groupby("ticker")["close"].transform("first") * 100

# dashboard visual theme
dashboard_font = "IBM Plex Sans, Segoe UI, Arial, sans-serif"
dashboard_layout = {
    "font": {"family": dashboard_font, "size": 13, "color": "#1f2937"},
    "plot_bgcolor": "#f8fafc",
    "paper_bgcolor": "#ffffff",
    "margin": {"l": 60, "r": 30, "t": 70, "b": 55}
}

# brand-inspired palette (adjusted to keep similar blues distinguishable)
ticker_colors = {
    "NVDA": "#76B900",
    "AMD": "#ED1C24",
    "ASML": "#005EB8",
    "ARM": "#00A0DF",
    "SOXX": "#6F42C1"
}
nvda_color = ticker_colors["NVDA"]
other_colors = {k: v for k, v in ticker_colors.items() if k != "NVDA"}

# showing fig 1 price compare
fig = px.line(
    df, x="date",
    y="norm_price",
    custom_data=["open", "close"],
    color="ticker",
    color_discrete_map=ticker_colors,
    title="<b>Normalized Stock Price Performance (Base = Jan 15, 2025 = 100)</b>",
    labels={
        "date": "Date",
        "norm_price": "Normalized Price",
        "ticker": "Ticker"
    },

)
fig.update_layout(**dashboard_layout, legend_title_text="Ticker")
fig.update_xaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig.update_traces(hovertemplate="Date: %{x|%m/%d/%Y}<br>Ticker: %{fullData.name}<br>Normalized Price: %{y:.2f}<br>Open: %{customdata[0]:.2f}<br>Close: %{customdata[1]:.2f}<extra></extra>")
fig.update_traces(selector=dict(name="NVDA"), line=dict(width=4))


## Public Attention vs. NVDA Price
### Scatter plot of Wikipedia daily pageviews against NVDA normalized price, colored by month with an overall OLS trendline.

In [49]:
# Load pageviews from local CSV instead of Wikimedia API
wiki = pd.read_csv(data_dir / "wikimedia_pageviews.csv")
wiki["date"] = pd.to_datetime(wiki["timestamp"].astype(str).str[:8])
df2 = wiki[["date", "views"]]

df_nvda = df[df["ticker"] == "NVDA"].copy()
df_merge = pd.merge(df_nvda, df2, on="date", how="inner")
df_merge = df_merge[(df_merge["date"] >= start) & (df_merge["date"] <= end)]
df_merge["month_label"] = df_merge["date"].dt.strftime("%B %Y")

month_order = (
    df_merge[["month_label", "date"]]
    .drop_duplicates()
    .sort_values("date")["month_label"]
    .tolist()
)
# Higher-contrast sequential blue shades so March remains easy to see
month_shades = ["#BFDBFE", "#60A5FA", "#1D4ED8", "#1E3A8A"]
month_colors = {m: month_shades[i % len(month_shades)] for i, m in enumerate(month_order)}

fig2 = px.scatter(
    df_merge,
    custom_data=["date"],
    x="norm_price",
    y="views",
    trendline="ols",
    trendline_scope="overall",
    title="<b>NVDA: Attention vs Stock Price</b>",
    labels={
        "views": "Wikipedia Pageviews",
        "norm_price": "Normalized NVDA Price (Jan 15 = 100)",
        "month_label": "Month",
        "ticker": "Ticker"
    },
    color="month_label",
    color_discrete_map=month_colors,
    category_orders={"month_label": month_order},
)



fig2.update_layout(**dashboard_layout, legend_title_text="Month")
fig2.update_xaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig2.update_yaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig2.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=12, opacity=0.95, line=dict(width=1.2, color="#334155")),
    hovertemplate="Date: %{customdata[0]|%m/%d/%Y}<br>Month: %{fullData.name}<br>Normalized Price: %{x:.2f}<br>Wikipedia Pageviews: %{y:,.0f}<extra></extra>"
)

In [50]:
# fig2.show()


## Price, Wikipedia Views & News Tone
### Three signals overlaid on a shared time axis — NVDA closing price (left axis), Wikipedia pageviews, and GDELT average news tone bars (right axis). DeepSeek event marked on Jan 27.

In [51]:
import dash
from dash import dcc, html
from dash import Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [52]:
gdelt = pd.read_csv(data_dir / "gdelt_gkg_tone_timeline.csv")
wiki_views = pd.read_csv(data_dir / "wikimedia_pageviews.csv")

gdelt["date"] = pd.to_datetime(gdelt["Date"])
# Normalize tone to one value per date in case multiple series exist
gdelt["tone"] = gdelt["Value"]
gdelt = gdelt.groupby("date", as_index=False)["tone"].mean()

wiki_views["date"] = pd.to_datetime(
    wiki_views["timestamp"].astype(str).str[:8], format="%Y%m%d"
 )
wiki_views = wiki_views[["date", "views"]]

df_nvda = df[df["ticker"] == "NVDA"][["date", "close", "norm_price", "ticker"]].copy()
df_fig3 = (
    df_nvda.merge(wiki_views, on="date", how="left")
           .merge(gdelt[["date", "tone"]], on="date", how="left")
)

df_fig3 = df_fig3[
    (df_fig3["date"] >= pd.to_datetime(start)) & (df_fig3["date"] <= pd.to_datetime(end))
]

tone_colors = df_fig3["tone"].fillna(0).apply(lambda x: "#1D9E75" if x >= 0 else "#D85A30")
views_max = df_fig3["views"].max()
views_min = df_fig3["views"].min()
tone_max = df_fig3["tone"].abs().max()
if tone_max > 0:
    tone_scaled_init = df_fig3["tone"] / tone_max * (views_max - views_min) * 0.3 + views_min
else:
    tone_scaled_init = df_fig3["tone"]


fig3 = make_subplots(specs=[[{"secondary_y": True}]])

fig3.add_trace(
    go.Scatter(
        x=df_fig3["date"],
        y=df_fig3["close"],
        name="NVDA Close",
        mode="lines",
        line=dict(color=nvda_color, width=4),
        hovertemplate="Date: %{x|%m/%d/%Y}<br>NVDA Close: %{y:.2f}<extra></extra>"
    ),
    secondary_y=False
)

fig3.add_trace(
    go.Scatter(
        x=df_fig3["date"],
        y=df_fig3["views"],
        name="Wikipedia Views",
        mode="lines",
        line=dict(color="blue", dash="dot"),
        hovertemplate="Date: %{x|%m/%d/%Y}<br>Wikipedia Pageviews: %{y:,.0f}<extra></extra>"
    ),
    secondary_y=True
)

fig3.add_trace(
    go.Bar(
        x=df_fig3["date"],
        y=tone_scaled_init,
        name="GDELT Tone",
        customdata=df_fig3["tone"],
        marker_color=tone_colors,
        opacity=0.5,
        hovertemplate="Date: %{x|%m/%d/%Y}<br>GDELT Tone: %{customdata:.3f}<extra></extra>"
    ),
    secondary_y=True
)

event_date = "2025-01-27"
fig3.add_shape(
    type="line",
    x0=event_date,
    x1=event_date,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="black", dash="dash")
)
fig3.add_annotation(
    x=event_date,
    y=1,
    xref="x",
    yref="paper",
    text="DeepSeek Jan 27",
    showarrow=False,
    xanchor="left",
    yanchor="bottom"
 )

fig3.update_layout(
    **dashboard_layout,
    title="<b>NVDA Price vs Wikipedia Views vs GDELT Tone</b>",
    hovermode="x unified"
 )

fig3.update_xaxes(title_text="Date", showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig3.update_yaxes(title_text="NVDA Close Price (USD)", secondary_y=False, showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig3.update_yaxes(title_text="Wikipedia Views / News Tone", secondary_y=True, showgrid=False)

## Dash App: Interactive Dashboard
### Single-screen Plotly Dash dashboard with three coordinated views. Controls: month dropdown and date range slider filter all charts simultaneously. Click any point to highlight that date across all views. Scroll to zoom on any chart. Run the final cell to launch at http://127.0.0.1:8050.
### Note: The required analysis window ends on 03/15/2025, but 03/15 is a non-trading day, so market-price series may display their last available trading date (03/14/2025).

In [57]:
from dash import callback_context, no_update
import numpy as np

# create a date slider on the top
all_dates = sorted(df["date"].unique())
idx_to_date = {i: d for i, d in enumerate(all_dates)}
n = len(all_dates)

marks = {i: pd.Timestamp(all_dates[i]).strftime('%b %d') for i in range(0, n, 7)}
marks[n-1] = pd.Timestamp(all_dates[-1]).strftime('%b %d')

# date range slider + month dropdown + reset button
app = dash.Dash(__name__, external_stylesheets=[
    'https://fonts.googleapis.com/css2?family=Inter:wght@400;500&display=swap'
])

app.layout = html.Div([
    html.H1("NVIDIA Signal Dashboard: Stock, Attention & News Tone (Jan-Mar 2025)", style={'textAlign': 'center', 'maxWidth': '980px', 'margin': '0 auto 8px auto', 'lineHeight': '1.16', 'fontSize': 'clamp(1.55rem, 3.2vw, 2.25rem)', 'fontWeight': '700', 'letterSpacing': '-0.02em'}),

    # adding a guide
    html.P([
        "Investigating whether NVDA's price movement in winter 2025 aligned with public attention, AI news sentiment, or broader semiconductor momentum.",
        html.Br(),
        "Note: 03/15/2025 is a non-trading day, so market-price lines may end at the last available trading day (03/14/2025)."
    ],
        style={
            'textAlign': 'center',
            'color': 'gray',
            'fontSize': '13px',
            'marginTop': '10px',
            'marginBottom': '32px'
        }
    ),

    html.Div([
        html.Div([
            html.Label("Month", style={'fontSize': '12px', 'color': 'gray', 'marginBottom': '0'}),
            dcc.Dropdown(
                id='month-selector',
                options=[
                    {'label': 'Jan', 'value': 'JAN'},
                    {'label': 'Feb', 'value': 'FEB'},
                    {'label': 'Mar', 'value': 'MAR'},
                    {'label': 'All', 'value': 'ALL'}
                ],
                value='ALL', clearable=False,
            )
        ], style={
            'width': '170px',
            'display': 'flex',
            'flexDirection': 'column',
            'gap': '6px'
        }),

        html.Div([
            html.Label("Date Range", style={'fontSize': '12px', 'color': 'gray', 'marginBottom': '0'}),
            dcc.RangeSlider(
                id='date-slider',
                min=0, max=n-1, step=1,
                value=[0, n-1],
                marks=marks,
                tooltip={"placement": "bottom", "always_visible": False},
                updatemode='drag'
            ),
        ], style={
            'flex': '1 1 620px',
            'minWidth': '320px',
            'maxWidth': '700px',
            'display': 'flex',
            'flexDirection': 'column',
            'gap': '6px'
        }),

        html.Button(
            "Reset",
            id='reset-btn',
            n_clicks=0,
            style={
                'height': '38px',
                'padding': '0 14px',
                'borderRadius': '8px',
                'border': '1px solid #cbd5e1',
                'backgroundColor': '#f8fafc',
                'color': '#0f172a',
                'fontWeight': '600',
                'cursor': 'pointer'
            }
        )
    ], style={
        'display': 'flex',
        'alignItems': 'flex-end',
        'justifyContent': 'center',
        'gap': '14px',
        'flexWrap': 'wrap',
        'maxWidth': '980px',
        'width': '100%',
        'margin': '8px auto 20px auto'
    }),

    dcc.Store(id='clicked-date', data=None),
    dcc.Store(id='zoom-state', data=None),

    html.Div([
        dcc.Graph(id='fig1', config={'scrollZoom': True}, style={'height': '420px', 'width': '100%'})
    ], style={'margin': '0 auto 20px auto', 'maxWidth': '980px', 'width': '100%'}),

    html.Div([
        dcc.Graph(id='fig2', config={'scrollZoom': True}, style={'height': '420px', 'width': '100%'})
    ], style={'margin': '0 auto 20px auto', 'maxWidth': '980px', 'width': '100%'}),

    html.Div([
        dcc.Graph(id='fig3', config={'scrollZoom': True}, style={'height': '420px', 'width': '100%'})
    ], style={'margin': '0 auto 20px auto', 'maxWidth': '980px', 'width': '100%'}),

], style={
    'minWidth': '0',
    'maxWidth': '1120px',
    'width': '100%',
    'margin': '0 auto',
    'padding': '0 24px 24px 24px',
    'fontFamily': 'Inter, sans-serif'
})

@app.callback(
    Output('date-slider', 'value'),
    [Input('reset-btn', 'n_clicks'),
     Input('month-selector', 'value')],
    prevent_initial_call=True
)
def update_slider(n_clicks, selected_period):
    ctx = callback_context
    trigger = ctx.triggered[0]['prop_id']
    if 'reset-btn' in trigger:
        return [0, n-1]
    month_map = {
        'ALL': ('2025-01-15', '2025-03-15'),
        'JAN': ('2025-01-15', '2025-01-31'),
        'FEB': ('2025-02-01', '2025-02-28'),
        'MAR': ('2025-03-01', '2025-03-15')
    }
    s, e = month_map.get(selected_period, ('2025-01-15', '2025-03-15'))
    s_ts, e_ts = pd.Timestamp(s), pd.Timestamp(e)
    s_idx = min(range(n), key=lambda i: abs(pd.Timestamp(idx_to_date[i]) - s_ts))
    e_idx = min(range(n), key=lambda i: abs(pd.Timestamp(idx_to_date[i]) - e_ts))
    return [s_idx, e_idx]

# Callback B: clicked date -> store in dcc.Store
@app.callback(
    Output('clicked-date', 'data'),
    [Input('fig1', 'clickData'),
     Input('fig2', 'clickData'),
     Input('fig3', 'clickData')],
    prevent_initial_call=True
)
def store_clicked_date(click1, click2, click3):
    ctx = callback_context
    trigger = ctx.triggered[0]['prop_id']
    if 'fig1' in trigger and click1:
        return click1['points'][0]['x']
    elif 'fig2' in trigger and click2:
        pt = click2['points'][0]
        if 'customdata' in pt:
            return pt['customdata'][0]
    elif 'fig3' in trigger and click3:
        return click3['points'][0]['x']
    return no_update

# Callback D: any chart zoom -> store zoom range
@app.callback(
    Output('zoom-state', 'data'),
    [Input('fig1', 'relayoutData'),
     Input('fig3', 'relayoutData')],
    prevent_initial_call=True
)
def store_zoom(relay1, relay3):
    ctx = callback_context
    trigger = ctx.triggered[0]['prop_id']
    relay = relay1 if 'fig1' in trigger else relay3
    if not relay:
        return no_update
    if 'xaxis.range[0]' in relay:
        return {'x0': relay['xaxis.range[0]'], 'x1': relay['xaxis.range[1]']}
    if 'xaxis.autorange' in relay:
        return None  # zoom reset
    return no_update

# Callback C: slider + clicked date -> update all 3 figures
@app.callback(
    [Output('fig1', 'figure'),
     Output('fig2', 'figure'),
     Output('fig3', 'figure')],
    [Input('date-slider', 'value'),
     Input('clicked-date', 'data'),
     Input('zoom-state', 'data')]
)
def update_all(slider_range, clicked_date, zoom_state):
    start_dt = idx_to_date[slider_range[0]]
    end_dt = idx_to_date[slider_range[1]]

    filtered_df = df[(df["date"] >= start_dt) & (df["date"] <= end_dt)]
    new_fig1 = px.line(
        filtered_df, x="date", y="norm_price", color="ticker",
        custom_data=["open", "close"],
        color_discrete_map=ticker_colors,
        title="<b>Semiconductor Peer Comparison: Normalized Price Performance (Jan 15 = 100)</b>",
        labels={"date": "Date", "norm_price": "Normalized Price", "ticker": "Ticker"}
    )
    new_fig1.update_layout(**dashboard_layout, legend_title_text="Ticker", height=420)
    new_fig1.update_xaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig1.update_yaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig1.update_traces(hovertemplate="Date: %{x|%m/%d/%Y}<br>Ticker: %{fullData.name}<br>Normalized Price: %{y:.2f}<br>Open: %{customdata[0]:.2f}<br>Close: %{customdata[1]:.2f}<extra></extra>")
    new_fig1.update_traces(selector=dict(name="NVDA"), line=dict(width=4))

    
    #adding a DeepSeek event line on Jan 27, 2025
    new_fig1.add_shape(
        type="line", x0="2025-01-27", x1="2025-01-27",
        y0=0, y1=1, xref="x", yref="paper",
        line=dict(color="gray", dash="dash", width=1)
    )
    new_fig1.add_annotation(
        x="2025-01-27", y=1, xref="x", yref="paper",
        text="DeepSeek", showarrow=False,
        xanchor="left", yanchor="bottom",
        font=dict(size=10, color="gray")
    )

    filtered_merge = df_merge[(df_merge["date"] >= start_dt) & (df_merge["date"] <= end_dt)]
    if len(filtered_merge) < 2:
        new_fig2 = px.scatter(
            filtered_merge, x="norm_price", y="views",
            color="month_label",
            color_discrete_map=month_colors,
            category_orders={"month_label": month_order},
            labels={
                "views": "Wikipedia Pageviews",
                "norm_price": "Normalized NVDA Price (Jan 15 = 100)",
                "month_label": "Month",
                "ticker": "Ticker"
            },
            custom_data=["date"],
            title="<b>Public Attention vs. Price (color: month)(not enough data)</b>"
        )
    else:
        new_fig2 = px.scatter(
            filtered_merge, x="norm_price", y="views",
            trendline="ols", trendline_scope="overall",
            color="month_label",
            color_discrete_map=month_colors,
            category_orders={"month_label": month_order},
            labels={
                "views": "Wikipedia Pageviews",
                "norm_price": "Normalized NVDA Price (Jan 15 = 100)",
                "month_label": "Month",
                "ticker": "Ticker"
            },
            custom_data=["date"],
            title="<b>Public Attention vs. Price (color: month)</b>"
        )
    new_fig2.update_layout(**dashboard_layout, legend_title_text="Month", height=420,
        hoverlabel=dict(bgcolor="white", font_color="black", bordercolor="gray")
    )
    if len(filtered_merge) >= 2:
        X_cb = sm.add_constant(filtered_merge["norm_price"])
        model_cb = sm.OLS(filtered_merge["views"], X_cb).fit()
        r2_cb = model_cb.rsquared
        new_fig2.add_annotation(
            x=0.98, y=0.95,
            xref="paper", yref="paper",
            text=f"R² = {r2_cb:.3f}",
            showarrow=False,
            xanchor="right",
            font=dict(size=13, color="#334155"),
            bgcolor="white",
            bordercolor="#e5e7eb",
            borderwidth=1
        )
    new_fig2.update_xaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig2.update_yaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig2.update_traces(
        selector=dict(mode="markers"),
        marker=dict(size=12, opacity=0.95, line=dict(width=1.2, color="#334155")),
        hovertemplate="Date: %{customdata[0]|%m/%d/%Y}<br>Month: %{fullData.name}<br>Normalized Price: %{x:.2f}<br>Wikipedia Pageviews: %{y:,.0f}<extra></extra>"
    )

    filtered_f3 = df_fig3[(df_fig3["date"] >= start_dt) & (df_fig3["date"] <= end_dt)]
    tone_clr = filtered_f3["tone"].fillna(0).apply(lambda x: "#1D9E75" if x >= 0 else "#D85A30")
    views_max = filtered_f3["views"].max()
    views_min = filtered_f3["views"].min()
    tone_max = filtered_f3["tone"].abs().max()
    if tone_max > 0:
        tone_scaled = filtered_f3["tone"] / tone_max * (views_max - views_min) * 0.3 + views_min
    else:
        tone_scaled = filtered_f3["tone"]

    new_fig3 = make_subplots(specs=[[{"secondary_y": True}]])
    new_fig3.add_trace(go.Scatter(
        x=filtered_f3["date"], y=filtered_f3["close"],
        name="NVDA Close", mode="lines", line=dict(color=nvda_color, width=4),
        hovertemplate="Date: %{x|%m/%d/%Y}<br>NVDA Close: %{y:.2f}<extra></extra>"
    ), secondary_y=False)
    new_fig3.add_trace(go.Scatter(
        x=filtered_f3["date"], y=filtered_f3["views"],
        name="Wikipedia Views", mode="lines", line=dict(color="blue", dash="dot"),
        hovertemplate="Date: %{x|%m/%d/%Y}<br>Wikipedia Pageviews: %{y:,.0f}<extra></extra>"
    ), secondary_y=True)
    new_fig3.add_trace(go.Bar(
        x=filtered_f3["date"], y=tone_scaled,
        name="GDELT Tone", customdata=filtered_f3["tone"], marker_color=tone_clr, opacity=0.5,
        hovertemplate="Date: %{x|%m/%d/%Y}<br>GDELT Tone: %{customdata:.3f}<extra></extra>"
    ), secondary_y=True)
    new_fig3.add_shape(
        type="line", x0="2025-01-27", x1="2025-01-27",
        y0=0, y1=1, xref="x", yref="paper",
        line=dict(color="black", dash="dash")
    )
    new_fig3.add_annotation(
        x="2025-01-27", y=1, xref="x", yref="paper",
        text="DeepSeek Jan 27", showarrow=False, xanchor="left", yanchor="bottom"
    )
    new_fig3.update_layout(
        **dashboard_layout,
        title="<b>Price, Wikipedia Views & News Tone</b>",
        hovermode="x unified",
        height=420
    )
    new_fig3.update_xaxes(title_text="Date", showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig3.update_yaxes(title_text="NVDA Close Price (USD)", secondary_y=False, showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    new_fig3.update_yaxes(title_text="Wikipedia Views / News Tone", secondary_y=True, showgrid=False)

    if clicked_date:
        clicked_ts = pd.Timestamp(clicked_date)
        new_fig1.add_shape(type="line", x0=clicked_ts, x1=clicked_ts, y0=0, y1=1, xref="x", yref="paper", line=dict(color="orange", dash="dash", width=2))
        new_fig1.add_annotation(x=clicked_ts, y=1, xref="x", yref="paper", text=str(clicked_ts.date()), showarrow=False, xanchor="left", yanchor="bottom", font=dict(color="orange"))
        match = filtered_merge[filtered_merge["date"] == clicked_ts]
        if not match.empty:
            new_fig2.add_trace(go.Scatter(
                x=match["norm_price"], y=match["views"],
                mode="markers",
                marker=dict(color="orange", size=24, symbol="diamond-open", line=dict(width=5)),
                name="Selected", showlegend=False
            ))
        new_fig3.add_shape(type="line", x0=clicked_ts, x1=clicked_ts, y0=0, y1=1, xref="x", yref="paper", line=dict(color="orange", dash="dash", width=2))

    if zoom_state:
        x0 = zoom_state['x0']
        x1 = zoom_state['x1']
        new_fig1.update_xaxes(range=[x0, x1])
        new_fig3.update_xaxes(range=[x0, x1])

    return new_fig1, new_fig2, new_fig3

In [58]:
if __name__ == "__main__":
    app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
